In [5]:
import pymysql
import datetime
from datetime import datetime
import csv

In [6]:
from db_config import get_connection  # credentials come from api/.env

conn = get_connection(autocommit=True)

cur = conn.cursor(pymysql.cursors.DictCursor)

In [12]:
cur.execute("DROP TABLE IF EXISTS hm_articles")
cur.execute("""
CREATE TABLE hm_articles (
    article_id BIGINT PRIMARY KEY,
    prod_name VARCHAR(255),
    product_type_name VARCHAR(255),
    product_group_name VARCHAR(255),
    department_name VARCHAR(255),
    index_name VARCHAR(255),
    colour_group_name VARCHAR(255),
    garment_group_name VARCHAR(255),
    detail_desc TEXT
)ENGINE=InnoDB;
""")

0

In [3]:
cur.execute("DROP TABLE IF EXISTS hm_customers;")
cur.execute("""
CREATE TABLE hm_customers (
    customer_id VARCHAR(64) PRIMARY KEY,
    FN INT,
    Active INT,
    club_member_status VARCHAR(50),
    fashion_news_frequency VARCHAR(50),
    age INT,
    postal_code CHAR(64)
) ENGINE=InnoDB;
""")

0

In [3]:
cur.execute("DROP TABLE IF EXISTS hm_transactions;")
cur.execute("""
CREATE TABLE hm_transactions (
    t_dat DATE,
    customer_id VARCHAR(64),
    article_id BIGINT,
    price DECIMAL(10,5),
    sales_channel_id INT,
    INDEX idx_customer (customer_id),
    INDEX idx_article (article_id)
) ENGINE=InnoDB;
""")

0

In [13]:
article_rows = []
with open("articles.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        article_rows.append((
            row["article_id"],
            row["prod_name"],
            row["product_type_name"],
            row["product_group_name"],
            row["department_name"],
            row["index_name"],
            row["colour_group_name"],
            row["garment_group_name"],
            row["detail_desc"]
        ))

cur.executemany("""
INSERT INTO hm_articles VALUES (
    %s,%s,%s,%s,%s,%s,%s,%s,%s
)
""", article_rows)

105542

In [4]:
import csv

def to_int_or_none(v):
    if v is None or v == 'NULL':
        return None
    v = str(v).strip()
    return int(v) if v.isdigit() else None

def to_null(v):
    if v is None:
        return None
    v = str(v).strip()
    return None if v == '' or v == 'NULL' else v

customer_rows = []

with open("customers.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        customer_rows.append((
            row["customer_id"],
            to_int_or_none(row["FN"]),
            to_int_or_none(row["Active"]),
            to_null(row["club_member_status"]),
            to_null(row["fashion_news_frequency"]),
            to_int_or_none(row["age"]),
            to_null(row["postal_code"])
        ))

cur.executemany("""
INSERT INTO hm_customers
(customer_id, FN, Active, club_member_status, fashion_news_frequency, age, postal_code)
VALUES (%s,%s,%s,%s,%s,%s,%s)
""", customer_rows)
conn.commit()

In [ ]:
batch = []
batch_size = 500000

with open("transactions.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        batch.append((
            row["t_dat"],
            row["customer_id"],
            row["article_id"],
            row["price"],
            row["sales_channel_id"]
        ))

        if len(batch) >= batch_size:
            cur.executemany("""
                INSERT INTO hm_transactions
                (t_dat, customer_id, article_id, price, sales_channel_id)
                VALUES (%s,%s,%s,%s,%s)
            """, batch)
            conn.commit()
            print("Inserted", len(batch))
            batch = []


if batch:
    cur.executemany("""
        INSERT INTO hm_transactions
        (t_dat, customer_id, article_id, price, sales_channel_id)
        VALUES (%s,%s,%s,%s,%s)
    """, batch)
    conn.commit()



Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000
Inserted 500000


In [9]:
cur.execute("SELECT COUNT(*) AS total FROM hm_articles;")
rows = cur.fetchall()
print(rows)



[{'total': 105542}]


In [11]:
cur.execute("SELECT COUNT(*) AS total FROM hm_customers;")
rows = cur.fetchall()
print(rows)



[{'total': 1371980}]


In [7]:
cur.execute("SELECT COUNT(*) AS total FROM hm_transactions;")
rows = cur.fetchall()
print(rows)



[{'total': 31788324}]


In [13]:
cur.execute("""
SELECT 
    t.customer_id, 
    t.article_id, 
    a.prod_name
FROM hm_transactions t
JOIN hm_articles a 
    ON t.article_id = a.article_id
LIMIT 10;
""")

rows = cur.fetchall()
print(rows)



[{'customer_id': '05ed96931b707698bc94aa53766d44686ae5ccbbc99dfbda8694ae811c54f28f', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '05ed96931b707698bc94aa53766d44686ae5ccbbc99dfbda8694ae811c54f28f', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '0aaa9683d5f45b85d0f9a81e2f4d4ef774fed43339fb75bab45673c6a581841a', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '20ccafd82d923baecf1fb8705d459e243c8ca9c74ee24e4adf0cb1fe2ae4a13a', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '2135790e6b63d10b59f9d8b84a58e24328b0f5ed3a48f15c0a29d4832fdcf167', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '3329634c9451f438049c5ebb4e77d0e0b7c730228aa330f581e5bdb2e566fc66', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '3e6b4fc03a858ef3d12e58d5b525e2f4bd88dea270161576c4c14931da571e48', 'article_id': 108775015, 'prod_name': 'Strap top'}, {'customer_id': '3e6b4fc03a858ef3d12e58d5b525e2

In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS article_summary AS
SELECT 
    article_id,
    COUNT(*) AS units_sold,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(price) AS revenue
FROM hm_transactions
GROUP BY article_id;
""")
conn.commit()


In [8]:
cur.execute("""
CREATE TABLE IF NOT EXISTS monthly_revenue AS
SELECT 
    DATE_FORMAT(t_dat, '%Y-%m') AS month,
    SUM(price) AS revenue
FROM hm_transactions
GROUP BY month
ORDER BY month;
""")
conn.commit()


In [5]:
cur.execute("""
SELECT 
    a.article_id,
    a.units_sold,
    a.unique_customers,
    a.revenue,
    b.prod_name
FROM article_summary a
JOIN hm_articles b 
    ON a.article_id = b.article_id
ORDER BY a.units_sold DESC
LIMIT 10;
""")
rows = cur.fetchall()
print(rows)


[{'article_id': 706016001, 'units_sold': 42667, 'unique_customers': 28139, 'revenue': Decimal('1379.48224'), 'prod_name': 'Jade HW Skinny Denim TRS'}, {'article_id': 706016002, 'units_sold': 31645, 'unique_customers': 23127, 'revenue': Decimal('1024.10525'), 'prod_name': 'Jade HW Skinny Denim TRS'}, {'article_id': 372860001, 'units_sold': 25508, 'unique_customers': 20890, 'revenue': Decimal('329.41080'), 'prod_name': '7p Basic Shaftless'}, {'article_id': 610776002, 'units_sold': 23789, 'unique_customers': 18063, 'revenue': Decimal('191.28544'), 'prod_name': 'Tilly (1)'}, {'article_id': 399223001, 'units_sold': 21030, 'unique_customers': 15177, 'revenue': Decimal('647.67235'), 'prod_name': 'Curvy Jeggings HW Ankle'}, {'article_id': 464297007, 'units_sold': 20959, 'unique_customers': 15830, 'revenue': Decimal('338.44026'), 'prod_name': 'Greta Thong Mynta Low 3p'}, {'article_id': 562245001, 'units_sold': 20110, 'unique_customers': 14719, 'revenue': Decimal('525.29230'), 'prod_name': 'Luna

In [4]:
cur.execute("""
SELECT 
    month,
    revenue
FROM monthly_revenue
ORDER BY month;
""")
rows = cur.fetchall()
print(rows)


[{'month': '2018-09', 'revenue': Decimal('17755.68754')}, {'month': '2018-10', 'revenue': Decimal('41584.97025')}, {'month': '2018-11', 'revenue': Decimal('39000.96209')}, {'month': '2018-12', 'revenue': Decimal('32532.77089')}, {'month': '2019-01', 'revenue': Decimal('33450.37010')}, {'month': '2019-02', 'revenue': Decimal('31428.57637')}, {'month': '2019-03', 'revenue': Decimal('37549.39593')}, {'month': '2019-04', 'revenue': Decimal('42825.32967')}, {'month': '2019-05', 'revenue': Decimal('43357.84236')}, {'month': '2019-06', 'revenue': Decimal('48648.85335')}, {'month': '2019-07', 'revenue': Decimal('40387.66491')}, {'month': '2019-08', 'revenue': Decimal('30546.13228')}, {'month': '2019-09', 'revenue': Decimal('40304.16699')}, {'month': '2019-10', 'revenue': Decimal('36988.10536')}, {'month': '2019-11', 'revenue': Decimal('38854.45122')}, {'month': '2019-12', 'revenue': Decimal('31267.80512')}, {'month': '2020-01', 'revenue': Decimal('28793.85277')}, {'month': '2020-02', 'revenue'

In [27]:
customer_id = "05ed96931b707698bc94aa53766d44686ae5ccbbc99dfbda8694ae811c54f28f"


In [29]:
cur.execute("""
SELECT 
    t.t_dat,
    t.article_id,
    a.prod_name,
    t.price,
    t.sales_channel_id
FROM hm_transactions t
JOIN hm_articles a 
    ON t.article_id = a.article_id
WHERE t.customer_id = %s
ORDER BY t.t_dat DESC
LIMIT 200;
""", (customer_id,))
rows = cur.fetchall()
print(rows)


[{'t_dat': datetime.date(2020, 3, 11), 'article_id': 608776002, 'prod_name': 'Scallop 5p Socks', 'price': Decimal('0.01354'), 'sales_channel_id': 1}, {'t_dat': datetime.date(2020, 2, 4), 'article_id': 803892001, 'prod_name': 'Rose rib tank', 'price': Decimal('0.01863'), 'sales_channel_id': 1}, {'t_dat': datetime.date(2020, 2, 4), 'article_id': 803892002, 'prod_name': 'Rose rib tank', 'price': Decimal('0.01863'), 'sales_channel_id': 1}, {'t_dat': datetime.date(2020, 1, 30), 'article_id': 636505001, 'prod_name': 'Shake it in shape swimsuit', 'price': Decimal('0.05083'), 'sales_channel_id': 2}, {'t_dat': datetime.date(2020, 1, 30), 'article_id': 636505001, 'prod_name': 'Shake it in shape swimsuit', 'price': Decimal('0.05083'), 'sales_channel_id': 2}, {'t_dat': datetime.date(2020, 1, 30), 'article_id': 811899001, 'prod_name': 'X-tina swimsuit', 'price': Decimal('0.04236'), 'sales_channel_id': 2}, {'t_dat': datetime.date(2020, 1, 30), 'article_id': 811899001, 'prod_name': 'X-tina swimsuit',

In [31]:
article_id = 108775015


In [33]:
cur.execute("""
SELECT 
    a.article_id,
    a.prod_name,
    a.product_type_name,
    a.colour_group_name,
    s.units_sold,
    s.unique_customers,
    s.revenue
FROM hm_articles a
LEFT JOIN article_summary s 
    ON a.article_id = s.article_id
WHERE a.article_id = %s;
""", (article_id,))
row = cur.fetchone()
print(row)


{'article_id': 108775015, 'prod_name': 'Strap top', 'product_type_name': 'Vest top', 'colour_group_name': 'Black', 'units_sold': 10836, 'unique_customers': 6880, 'revenue': Decimal('88.25661')}


In [13]:
###Capstone

# Add the business columns to hm_customers

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN customer_lifetime_value DECIMAL(12,2)
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN cohort_month VARCHAR(7)
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN days_since_last_purchase INT
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN total_transactions INT
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN first_purchase_date DATE
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN last_purchase_date DATE
""")

cur.execute("""
ALTER TABLE hm_customers ADD COLUMN churn_risk_score FLOAT
""")

conn.commit()
print("✓ All columns added successfully")

✓ All columns added successfully


In [14]:
# PHASE 2: Populate columns ONE AT A TIME

print("Populating customer_lifetime_value...")
cur.execute("""
UPDATE hm_customers c
SET customer_lifetime_value = (
    SELECT COALESCE(SUM(price), 0) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
)
""")
conn.commit()
print("✓ customer_lifetime_value done")

print("Populating total_transactions...")
cur.execute("""
UPDATE hm_customers c
SET total_transactions = (
    SELECT COUNT(*) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
)
""")
conn.commit()
print("✓ total_transactions done")

print("Populating first_purchase_date...")
cur.execute("""
UPDATE hm_customers c
SET first_purchase_date = (
    SELECT MIN(t_dat) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
)
""")
conn.commit()
print("✓ first_purchase_date done")

print("Populating last_purchase_date...")
cur.execute("""
UPDATE hm_customers c
SET last_purchase_date = (
    SELECT MAX(t_dat) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
)
""")
conn.commit()
print("✓ last_purchase_date done")

print("Populating days_since_last_purchase...")
cur.execute("""
UPDATE hm_customers c
SET days_since_last_purchase = DATEDIFF(CURDATE(), (
    SELECT MAX(t_dat) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
))
""")
conn.commit()
print("✓ days_since_last_purchase done")

print("Populating cohort_month...")
cur.execute("""
UPDATE hm_customers c
SET cohort_month = DATE_FORMAT((
    SELECT MIN(t_dat) 
    FROM hm_transactions 
    WHERE customer_id = c.customer_id
), '%Y-%m')
""")
conn.commit()
print("✓ cohort_month done")

print("\n✓✓✓ ALL COLUMNS POPULATED ✓✓✓")

Populating customer_lifetime_value...
✓ customer_lifetime_value done
Populating total_transactions...
✓ total_transactions done
Populating first_purchase_date...
✓ first_purchase_date done
Populating last_purchase_date...
✓ last_purchase_date done
Populating days_since_last_purchase...
✓ days_since_last_purchase done
Populating cohort_month...
✓ cohort_month done

✓✓✓ ALL COLUMNS POPULATED ✓✓✓
